# Lab 13: Regression Discontinuity — Diagnosing a Jump
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnostic Lab | 120 min Core + 30 min Extension

---

**Learning Objectives:**
- Estimate a discontinuity and show the estimate is a bandwidth choice as much as a finding
- Run the two tests that can refute an RD design, and say what passing them does not prove
- Detect manipulation of the running variable with a density test
- State the population an RD estimate belongs to, and why it is the narrowest in the course
- Compute a local linear RD estimate by hand, with a triangular kernel, and match `rdrobust` to every digit

**Dataset:** a simulated grant with a known effect, plus the US House data of Lee, Moretti and Butler (2004), which uses the close-election design of Lee (2008)

**Format:** Each diagnosis shows you an estimate or a test and asks what it does and does not tell you. Write your answers in the text cell under each set of questions.

**Deliverable:** your written diagnoses (Diagnoses 1, 2 and 3), the covariate test in Diagnosis 2, the required `rd_by_hand()` function, and `src/rd_tools.py`, a finished module you save and upload.

**Time:** Core (120 min, including the required Write it yourself) + Extension (optional, 30 min)

With setup, the README and the GitHub submission, plan on about two and a half hours, most of it after class. The optional Extension adds about 30 minutes.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI-assisted coding section: every line you add is yours, including the required "Write it yourself" function. The one AI use is the README prompt at the end, which asks for documentation, not code.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Diagnosis 2:** the covariate continuity test: a copy of the given covariate with a jump planted at the cutoff, and an `rdrobust` call on each of the two, about five lines in an empty cell.
> - **Write it yourself (required, after Diagnosis 3):** finish an `rd_by_hand()` function, a local linear RD estimate with a triangular kernel. Its first two lines are given; you write about five more. A check cell tells you when it is right. It is required.
> - **Extension (optional):** a fuzzy RD simulation, from a three-step outline.
>
> **You write, in text cells:** three answers in Diagnosis 1, three in Diagnosis 2 and two in Diagnosis 3.
>
> **You run and read:** everything else. That is the grant simulation, the analyst's global quadratic, the bandwidth table and its 300 draws, the two density tests and the given covariate, the House data and its density test, and Ship It's finished module, `rd_tools.py`, which is working code to read, not to retype. Read each cell before you run it: the questions ask about what it prints, and Write it yourself rebuilds Diagnosis 1's local linear estimate with `rdrobust`'s kernel.

In [ ]:
# GIVEN — run as-is
# Setup: install the two RD packages Colab does not have, then load the libraries
!pip install rdrobust==2.0.0 rddensity==3.0 -q

from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from rdrobust import rdrobust        # local linear RD estimates with a data-driven bandwidth
from rddensity import rddensity      # the density test for manipulation of the running variable

Path("src").mkdir(exist_ok=True)     # Ship It writes its module into this folder

rng = np.random.default_rng(seed=42)    # one generator for the lab's simulated data
print("Setup complete.")

## Diagnosis 1: The Estimate That Is a Bandwidth (35 min)

A grant is awarded to students whose need index is **50 or above**. The true effect on enrolment
is 8 percentage points, and you know that because you are about to simulate it. An analyst who
does not know it fits one quadratic to all 4,000 students, with a jump at the cutoff, a slope
that may change there and one curvature term for both sides, and reports **3.6 points**: less
than half the truth. After the analyst's estimate come the local linear estimate from
`rdrobust`, and then the estimate at seven bandwidths you choose by hand.

**Reading the code.** `gen.uniform(0, 100, n)` draws n numbers spread evenly between 0 and 100,
and `gen.normal(0, 0.12, n)` draws n normal errors with standard deviation 0.12; `4_000` is
4000 (the underscore only helps you read it). `(x >= 50).astype(int)` turns True/False into
1/0. `np.where(xc < 0, a, b)` takes `a` where `xc` is below 0 and `b` elsewhere, so the two
sides of the cutoff curve differently (−0.00006 · xc² below it, +0.00003 · xc² above).
`pd.DataFrame({...})` builds a table from named columns.

In [ ]:
# GIVEN — run as-is
# A grant at a need index of 50, with a true effect of 0.08 on enrolment
TRUE_JUMP = 0.08
n = 4_000


def simulate_grant(gen, n=n):
    """One sample of n students: need index x, the grant (treat) and enrolment y."""
    x = gen.uniform(0, 100, n)
    treat = (x >= 50).astype(int)      # treated ABOVE the cutoff
    xc = x - 50                        # distance from the cutoff
    base = 0.40 + 0.002 * xc + np.where(xc < 0, -0.00006 * xc**2, 0.00003 * xc**2)
    y = base + TRUE_JUMP * treat + gen.normal(0, 0.12, n)
    return pd.DataFrame({"x": x, "y": y, "treat": treat})


rd = simulate_grant(rng)

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# One quadratic for all 4,000 students: a jump, a slope that may change at 50, one curvature term
xc = rd["x"] - 50
X_poly = sm.add_constant(pd.DataFrame(
    {"treat": rd["treat"], "xc": xc, "xc2": xc**2, "treat_xc": rd["treat"] * xc}))
poly = sm.OLS(rd["y"], X_poly).fit()
print(f"Global quadratic estimate : {poly.params['treat']:+.4f}   truth {TRUE_JUMP:+.4f}")

**Reading the code.** `sm.add_constant(X)` adds a column of ones, the intercept, and
`sm.OLS(y, X).fit()` fits the regression; `.params["treat"]` is the coefficient on `treat`,
the jump. In the next cell, `rdrobust(y=..., x=..., c=50)` fits a line on each side of the
cutoff `c`, inside a bandwidth it chooses itself. Treatment here is *above* the cutoff, and
`rdrobust` reports the jump as the limit from the right minus the limit from the left, so the
sign comes out as you would read it. Its results are small tables with three rows.
"Conventional" is the local linear estimate. "Robust" is bias-corrected inference: it subtracts
an estimate of the bias and widens the standard error to allow for that, so its confidence
interval is centred on the bias-corrected estimate, not on the conventional one.
`.coef.loc["Conventional", "Coeff"]` is the estimate and `.bws.loc["h", "left"]` the
bandwidth, the distance from the cutoff on each side.

In [ ]:
# GIVEN — run as-is
# Local linear, with the bandwidth rdrobust chooses
grant_fit = rdrobust(y=rd["y"], x=rd["x"], c=50)
print(f"Local linear (rdrobust)   : {grant_fit.coef.loc['Conventional', 'Coeff']:+.4f}   "
      f"bandwidth {grant_fit.bws.loc['h', 'left']:.1f}")

**Expected output:** the analyst's global quadratic gives +0.0357 against a truth of +0.0800;
`rdrobust` gives +0.0789, at a bandwidth of 16.1.

**The estimate at seven bandwidths.** The next cell keeps only the students within h of the
cutoff and fits a line on each side, every student in the window weighted equally.
`local_linear` does one such fit: `treat_xc` lets the slope differ above the cutoff, and because
`xc` is measured from 50, the coefficient on `treat` is the gap between the two lines *at* 50.
`np.abs(rd["x"] - 50) <= h` is True for the rows within h of 50, and `rd[...]` keeps those rows.
`.bse` holds the standard errors, `len(band)` counts the rows, and `{b:>+11.4f}` right-aligns
`b` in 11 characters, with its sign and 4 decimals.

In [ ]:
# GIVEN — run as-is
# The estimate as a function of the bandwidth h


def local_linear(band):
    """A line on each side of 50, fitted to the rows in band with equal weights."""
    xc = band["x"] - 50
    X = sm.add_constant(pd.DataFrame(
        {"treat": band["treat"], "xc": xc, "treat_xc": band["treat"] * xc}))
    return sm.OLS(band["y"], X).fit()


GRID = [3, 5, 8, 12, 20, 35, 50]
print(f"{'bandwidth':>10}{'n':>8}{'estimate':>11}{'std err':>10}   truth {TRUE_JUMP:+.3f}")
print("-" * 52)
for h in GRID:
    band = rd[np.abs(rd["x"] - 50) <= h]
    fit = local_linear(band)
    print(f"{h:>10}{len(band):>8}{fit.params['treat']:>+11.4f}{fit.bse['treat']:>10.4f}")

**One draw cannot separate bias from variance; 300 draws can.** The next cell repeats the
simulation 300 times, each time with a new seed, and fits every bandwidth to every draw. For
each bandwidth, `bias` is the mean of the 300 estimates minus the truth, `MC sd` is their
standard deviation (`ddof=1` makes it the sample standard deviation), and RMSE, the root mean
squared error, is √(bias² + sd²). `draws` is a dictionary with one list per bandwidth;
`.append(x)` adds `x` to the end of a list. It takes a few seconds.

In [ ]:
# GIVEN — run as-is
# 300 samples of the same grant, each fitted at every bandwidth
REPS = 300
draws = {h: [] for h in GRID}           # one empty list of estimates per bandwidth

for r in range(REPS):
    sample = simulate_grant(np.random.default_rng(1000 + r))
    for h in GRID:
        band = sample[np.abs(sample["x"] - 50) <= h]
        draws[h].append(local_linear(band).params["treat"])

print(f"{'bandwidth':>10}{'mean est':>10}{'bias':>10}{'MC sd':>9}{'RMSE':>9}{'bias % of truth':>17}")
print("-" * 65)
for h in GRID:
    a = np.array(draws[h])
    bias = a.mean() - TRUE_JUMP
    rmse = np.sqrt(bias**2 + a.var(ddof=1))
    print(f"{h:>10}{a.mean():>10.4f}{bias:>+10.4f}{a.std(ddof=1):>9.4f}"
          f"{rmse:>9.4f}{100 * bias / TRUE_JUMP:>+16.1f}%")

**Expected output:** the bias runs from −0.0003 at h = 3 to −0.0382 at h = 50, and the MC sd
from 0.0309 down to 0.0071.

### YOUR DIAGNOSIS

1. The analyst's global quadratic (+0.0357) and the local linear estimate (+0.0789) disagree. Explain why a fit that uses students far from the cutoff can move the estimate *at* the cutoff. Compare the analyst's number with the h = 50 row of the bandwidth table, which also uses every student.
2. Read the Monte Carlo table as a bias-variance tradeoff. Name the bandwidth on the grid with the lowest RMSE, and explain why neither end of the grid is the answer.
3. No bandwidth is correct in general. On this draw `rdrobust` chose 16.1. It weights the students nearest the cutoff most (a triangular kernel), while the table weights every student in the window equally (a uniform kernel), so its 16.1 and the table's best row answer different questions. Say what its data-driven selector is optimising, and why its choice is a choice rather than a discovery.

*Your diagnosis here:*

---

## Diagnosis 2: The Two Tests That Can Refute You (30 min)

RD is the most credible observational design because, unusually, it has testable implications.

**Test 1: manipulation.** The density test (McCrary 2008; Cattaneo, Jansson and Ma 2020) asks
whether the density of the running variable jumps at the cutoff.

**Reading the code.** `rddensity(X=..., c=50)` runs the test on an array of running-variable
values; `.to_numpy()` turns a column into that array. `.test["t_jk"]` is the test statistic,
the jump in the density at the cutoff divided by its standard error, read like a z-statistic,
and `.test["p_jk"]` is its p-value.

In [ ]:
# GIVEN — run as-is
# Test 1: the density test on the grant data
dens = rddensity(X=rd["x"].to_numpy(), c=50)
print("Density test on clean data:")
print(f"   test statistic {dens.test['t_jk']:+.3f}, p-value {dens.test['p_jk']:.3f}")

The next cell draws a new population of 4,000 in which units can sort: 40% of those who land
just above the cutoff (50 to 56) move to just below it (45 to 50), as they would if being above
50 carried a cost. `rng.random(n) < 0.4` is True for about 40% of rows, `&` is "and" for arrays
of True/False, `.sum()` counts the Trues, and `xs[move] = ...` replaces only the rows where
`move` is True.

In [ ]:
# GIVEN — run as-is
# Test 1, continued: the density test on a population that sorts across the cutoff
xs = rng.uniform(0, 100, n)
just_above = (xs >= 50) & (xs < 56)
move = just_above & (rng.random(n) < 0.4)
xs[move] = rng.uniform(45, 50, move.sum())

dens_sorted = rddensity(X=xs, c=50)
print("Density test on manipulated data:")
print(f"   test statistic {dens_sorted.test['t_jk']:+.3f}, p-value {dens_sorted.test['p_jk']:.4f}")

**Expected output:** on the clean data the test statistic is +0.044 (p-value 0.965); on the
manipulated data it is −3.065 (p-value 0.0022).

**Test 2: covariate continuity.** A covariate measured *before* treatment, such as a student's
prior test score, cannot respond to the grant, so it should not jump at 50. The test is the same
RD with the covariate as the outcome. The next cell adds a prior score to `rd` that falls
smoothly as need rises, with noise whose standard deviation is 5 points.

In [ ]:
# GIVEN — run as-is
# Test 2: a prior test score, fixed before the grant, falling smoothly as need rises
rd["prior_score"] = 70 - 0.2 * rd["x"] + rng.normal(0, 5, len(rd))

**Your task.** In the empty cell below:
1. Add a second column, `prior_score_jump`: `prior_score` plus a jump at the cutoff about as
   large as the noise's standard deviation, 5 points.
2. Run `rdrobust` with each of the two columns as the outcome, `x` as the running variable and
   the cutoff at 50. Print each estimate and its robust p-value, `.pv.loc["Robust", "P>|z|"]`.

In [ ]:
# YOUR TASK — Test 2: plant a jump in a copy of prior_score, then test both columns

### YOUR DIAGNOSIS

1. The manipulation test looks only at the density of the running variable, not at the outcome. Explain why that is the right thing to examine. Diagnosis 3 runs the same test on a Democratic vote share at 50%: say what a failure there would mean.
2. Report your covariate test: the estimate and robust p-value for `prior_score` and for `prior_score_jump`. If a genuine pre-treatment covariate jumped at a real cutoff, what would that imply about the design?
3. Both tests pass on the clean data. State clearly what that does and does not establish.

*Your diagnosis here:*

---

## Diagnosis 3: A Real One (15 min)

The Lee (2008) design uses the 50% vote-share threshold in US House elections. The dataset below
is the one built on that design by **Lee, Moretti and Butler (2004), "Do Voters Affect or Elect
Policies? Evidence from the U.S. House", *QJE* 119(3)**, and it answers a different question
from Lee (2008)'s incumbency advantage.

**The outcome is `score`: the representative's inflation-adjusted ADA score** (Americans for
Democratic Action), the share of ADA-selected roll-call votes on which the member voted the
liberal way, adjusted across congresses by the Groseclose-Levitt-Snyder (1999) procedure. It
runs from -27.9 to 128.9 in this sample rather than 0 to 100, because the adjustment rescales
raw scores onto a common metric and can push extreme members outside the original range.
**Higher means more liberal.**

So the RD estimate below is measured in ADA points: how much more liberally a district is
represented when the Democrat *barely* wins rather than barely loses. It is a test of the
**partisan divergence** hypothesis (politicians pursue their own platform) against
**median-voter convergence** (a district gets the same policy either way). It is *not* an
incumbency advantage.

**Reading the code.** `pd.read_stata(url)` reads a Stata file from the web, and
`.dropna(subset=[...])` drops the rows missing either of those two columns. `{n:,}` prints a
whole number with commas. `rdrobust` prints "Mass points detected in the running variable."
when many rows share the same value of the running variable (here 13,577 rows have 6,121
distinct vote shares), and adjusts for them. A Democratic share of 0.5 + h is a margin of
victory of 2h, since the Republican has the rest. The second cell runs Diagnosis 2's density
test on the vote share, at 0.5.

In [ ]:
# GIVEN — run as-is
# Lee, Moretti and Butler (2004): US House, 1946-1995
lee = pd.read_stata(
    "https://raw.githubusercontent.com/scunning1975/mixtape/master/lmb-data.dta")
lee = lee.dropna(subset=["demvoteshare", "score"])
print(f"n = {len(lee):,} district-congress observations")
print(f"score (adjusted ADA): min {lee['score'].min():.1f}, "
      f"median {lee['score'].median():.1f}, max {lee['score'].max():.1f}")

house_fit = rdrobust(y=lee["score"], x=lee["demvoteshare"], c=0.5)
house_h = house_fit.bws.loc["h", "left"]
print(f"\nRD estimate at the 50% Democratic vote-share threshold: "
      f"{house_fit.coef.loc['Conventional', 'Coeff']:+.3f} ADA points")
print(f"Bandwidth: {house_h:.4f} vote share (Democratic share within {100 * house_h:.1f} points "
      f"of 50%, margins under about {200 * house_h:.0f} points)")

In [ ]:
# GIVEN — run as-is
# The density test on the Democratic vote share, at 0.5
dens_house = rddensity(X=lee["demvoteshare"].to_numpy(dtype=float), c=0.5)
print("Density test at a vote share of 0.5:")
print(f"   test statistic {dens_house.test['t_jk']:+.3f}, p-value {dens_house.test['p_jk']:.3f}")

**Expected output:** n = 13,577; the RD estimate is +46.491 ADA points at a bandwidth of
0.0856; the density test gives a test statistic of +0.363 (p-value 0.717).

### YOUR DIAGNOSIS

1. Name the population this estimate applies to, precisely. Would you use it to forecast a 20-point district?
2. Put the +46 ADA points on the scale of `score` printed above, and say which of the two hypotheses in the introduction it supports, and why.

*Your diagnosis here:*

---

## Write it yourself (required — 30 min)

This is the one function in this lab you write from an empty cell, and it is required. No AI:
write it yourself.

Diagnosis 1's bandwidth table fitted a line on each side of the cutoff with every student in the
window weighted equally: a uniform kernel. `rdrobust` uses a triangular kernel. A row at
distance |x − c| from the cutoff c gets the weight 1 − |x − c| / h, so the rows nearest the
cutoff count most, a row at the edge of the bandwidth counts for nothing, and rows farther than
h from the cutoff are left out.

In the cell below, finish the function `rd_by_hand(df, outcome, running, cutoff, h)` so it
returns the local linear RD estimate with a triangular kernel and bandwidth h, as one number.
`outcome` and `running` are column names, and a row is treated when its running variable is at
or above `cutoff`. Report the jump as `rdrobust` does: the limit from the right of the cutoff
minus the limit from the left. Use `statsmodels`, not `rdrobust`. The first two lines are given:
`dist` is each row's distance from the cutoff, and `inside` is True for the rows within h of it.
Replace the `____` with your own lines.

`sm.WLS(y, X, weights=w).fit()` is weighted least squares: like `sm.OLS(y, X).fit()`, but row
i's squared residual counts `w[i]` times. Its `.params` work as `sm.OLS`'s do. `np.abs(s)` (or
`s.abs()`) is the absolute value of every entry of `s`.

You need about five more lines. The check runs it at h = 10 on the grant data, where it
should reproduce `rdrobust`'s conventional estimate to every digit, and at the bandwidth
`rdrobust` chose in Diagnosis 1, where it should give the +0.0789 printed there. It then runs
it on half the sample, with the columns renamed and the cutoff moved, and on the House data of
Diagnosis 3, where it should give +46.491. Until you write the cell, the check says "Not written
yet"; once you do, it fails with a message until your function is right.

In [ ]:
# WRITE IT YOURSELF — required: finish rd_by_hand
def rd_by_hand(df, outcome, running, cutoff, h):
    """Local linear RD with a triangular kernel: the limit from the right minus the left."""
    dist = df[running] - cutoff          # each row's distance from the cutoff
    inside = dist.abs() <= h             # True for the rows within the bandwidth
    # Your lines: which rows are treated, X, the triangular weights, sm.WLS, and the return
    return ____

In [ ]:
# CHECK — for rd_by_hand above. Run it; you need not read it.
# globals() holds every name defined so far in this notebook
def names_used(code):
    """Every global name a function's code refers to, including inside functions it defines."""
    names = set(code.co_names)
    for const in code.co_consts:
        if hasattr(const, "co_names"):
            names |= names_used(const)
    return names


BLANK = "_" * 4                  # four underscores, the blank in the given cell


if "rd_by_hand" not in globals() or BLANK in names_used(rd_by_hand.__code__):
    print(f"Not written yet: replace the {BLANK} in rd_by_hand above, then run this check again.")
else:
    import inspect

    def package(df, outcome, running, cutoff, h):
        """rdrobust's conventional estimate at bandwidth h, with its triangular kernel."""
        fit = rdrobust(y=df[outcome], x=df[running], c=cutoff, h=h, kernel="triangular")
        return float(fit.coef.loc["Conventional", "Coeff"])

    def same(a, b):
        """Equal to about eight significant digits: only rounding inside the computer passes."""
        return bool(np.isclose(a, b, rtol=1e-8, atol=0))

    def rounded(a, b):
        """True if a is b rounded to some number of decimals, but not b itself."""
        return not same(a, b) and any(a == round(b, k) for k in range(7))

    ARGUMENT_OF = {"y": "outcome", "x": "running", "score": "outcome", "demvoteshare": "running"}

    def run(where, df, outcome, running, cutoff, h, on_error=""):
        """Call rd_by_hand; return its result as a number, or stop with a plain message."""
        try:
            got = rd_by_hand(df, outcome, running, cutoff, h)
        except KeyError as err:
            name = err.args[0] if err.args else ""
            if isinstance(name, int):
                hint = (f" .params[{name}] picks a coefficient by its position. Pick it by the name "
                        "of its column in X.")
            elif name == "treat":
                hint = (" Only the grant data has a treat column: build the treated rows from the "
                        "running variable and the cutoff.")
            elif name in ARGUMENT_OF and name not in (outcome, running):
                hint = f" Does it use its {ARGUMENT_OF[name]} argument, or '{name}'?"
            else:
                hint = on_error
            raise AssertionError(f"{where}, rd_by_hand stopped with KeyError {err}.{hint}") from None
        except np.linalg.LinAlgError as err:
            raise AssertionError(
                f"{where}, the regression in rd_by_hand stopped ({err}). Are some weights below "
                "zero? Inside the bandwidth each weight is 1 - |x - c| / h, between 0 and 1, and "
                "rows outside it are left out.") from None
        except Exception as err:
            text = str(err)
            if "dtype of object" in text:
                hint = (" Is one of the columns of X a True/False column? Turn it into 1/0, as "
                        "simulate_grant does with .astype(int).")
            elif "not aligned" in text or "Unalignable" in text:
                hint = (" Are y, X and the weights all the rows within h? Does it use df, not rd?")
            else:
                hint = on_error
            raise AssertionError(f"{where}, rd_by_hand stopped with {type(err).__name__}: "
                                 f"{text.rstrip('.')}.{hint}") from None
        assert got is not None, "rd_by_hand gives back nothing: use return, not print."
        assert not hasattr(got, "params"), (
            "rd_by_hand returns a fitted regression. Return the jump only, as a number.")
        try:
            value = float(got)
        except (TypeError, ValueError):
            raise AssertionError(f"rd_by_hand returns a {type(got).__name__}, not one number. "
                                 "Return the jump only.") from None
        assert np.isfinite(value), f"{where}, rd_by_hand gives {value}.{on_error}"
        return value

    assert len(rd) == 4000 and {"x", "y", "treat"} <= set(rd.columns), (
        "rd is no longer Diagnosis 1's grant data. Give the names in your cell other names, "
        "then rerun the notebook from the top.")
    # inspect.signature lists the names of a function's arguments, in order
    params = list(inspect.signature(rd_by_hand).parameters)
    assert params[:5] == ["df", "outcome", "running", "cutoff", "h"], (
        f"rd_by_hand's arguments are ({', '.join(params)}). The check calls it as the task asks: "
        "rd_by_hand(df, outcome, running, cutoff, h).")
    assert "rdrobust" not in names_used(rd_by_hand.__code__), (
        "rd_by_hand calls rdrobust. Write it with sm.WLS.")

    # The grant data at h = 10
    got10 = run("On the grant data at h = 10", rd.copy(), "y", "x", 50, 10)
    want10 = package(rd, "y", "x", 50, 10)
    assert not same(got10, -want10), (
        f"rd_by_hand gives {got10:+.4f} at h = 10: the right size with the wrong sign. The jump "
        "is the limit from the right of the cutoff minus the limit from the left.")
    slips = {
        0.48549833: "is the intercept of the right side alone: the level of enrolment just above "
                    "the cutoff. The jump is the right side's intercept minus the left side's",
        0.39788600: "is the intercept of the left side alone. The jump is the right side's "
                    "intercept minus the left side's",
        0.07680632: "comes from a running variable that was not measured from the cutoff: with x "
                    "itself, the gap between the two lines is taken at x = 0, not at the cutoff",
        0.08762336: "comes from one slope for both sides. Let the slope differ above the cutoff",
        0.08020003: "is the uniform kernel of Diagnosis 1's table, every row in the bandwidth "
                    "weighted equally. Weight each row by 1 - |x - c| / h",
        0.08793407: "uses the weight 1 - u squared. The triangular weight is 1 - |x - c| / h",
        0.08271813: "uses 1 - (x - c) / h without the absolute value, so rows below the cutoff "
                    "get weights above 1",
        0.08565273: "weights each row by the square root of its kernel weight. Pass the weights "
                    "themselves to sm.WLS",
        0.09383485: "compares weighted means on the two sides, with no slope. Fit a line on each side",
        0.03583456: "uses every row with equal weights: Diagnosis 1's h = 50 estimate. Keep only "
                    "the rows within h of the cutoff, and weight them",
    }
    for value, cause in slips.items():
        assert not np.isclose(got10, value, rtol=1e-6, atol=0), (
            f"On the grant data at h = 10, rd_by_hand gives {got10:.6f} and rdrobust "
            f"{want10:.6f}. Yours {cause}.")
    assert not rounded(got10, want10), (
        f"rd_by_hand gives {got10}, and rdrobust {want10}. Is the estimate rounded? "
        "Return it unrounded.")
    assert same(got10, want10), (
        f"On the grant data at h = 10, rd_by_hand gives {got10:.6f}; rdrobust gives "
        f"{want10:.6f}. Compare each step with the task above.")

    # Diagnosis 1's bandwidth
    grant_h = float(grant_fit.bws.loc["h", "left"])
    got_h = run(f"At Diagnosis 1's bandwidth of {grant_h:.1f}", rd.copy(), "y", "x", 50, grant_h)
    assert not same(got_h, got10), (
        f"At a bandwidth of {grant_h:.1f}, rd_by_hand gives its h = 10 answer. Does it use its h "
        "argument, or 10?")
    want_h = float(grant_fit.coef.loc["Conventional", "Coeff"])
    assert same(got_h, want_h), (
        f"At a bandwidth of {grant_h:.1f}, rd_by_hand gives {got_h:.6f}; Diagnosis 1's rdrobust "
        f"gave {want_h:.6f}.")

    # Does the function use each argument? Half the sample, other names and cutoff, real data
    half = rd.iloc[::2].copy()          # every second row
    got_half = run("On half the grant data", half, "y", "x", 50, 10)
    assert not same(got_half, got10), (
        "On half the grant data, rd_by_hand gives the answer for all 4,000 students. Does it "
        "use its df argument, or rd?")
    want_half = package(half, "y", "x", 50, 10)
    assert same(got_half, want_half), (
        f"On half the grant data, rd_by_hand gives {got_half:.6f}; rdrobust gives {want_half:.6f}.")

    moved = pd.DataFrame({"enrolled": rd["y"], "need_gap": rd["x"] - 50})   # cutoff now at 0
    got_moved = run("With the columns renamed (enrolled, need_gap) and the cutoff at 0", moved,
                    "enrolled", "need_gap", 0, 10,
                    on_error=" Does it use its outcome, running and cutoff arguments everywhere?")
    assert same(got_moved, want10), (
        f"With the columns renamed and the cutoff at 0, rd_by_hand gives {got_moved:.6f} "
        f"instead of {want10:.6f}. Does it use its cutoff argument, or 50?")

    house = lee[["score", "demvoteshare"]].astype(float)
    got_house = run("On the House data of Diagnosis 3", house, "score", "demvoteshare", 0.5,
                    float(house_h), on_error=" Does it use its df, cutoff and h arguments?")
    want_house = float(house_fit.coef.loc["Conventional", "Coeff"])
    assert same(got_house, want_house), (
        f"On the House data, rd_by_hand gives {got_house:.4f}; Diagnosis 3's rdrobust gave "
        f"{want_house:.4f}.")
    print(f"rd_by_hand matches rdrobust on the grant data at h = 10: {got10:+.4f}.")
    print(f"At Diagnosis 1's bandwidth of {grant_h:.1f}: {got_h:+.4f}.")
    print(f"On the House data at 0.5: {got_house:+.3f} ADA points.")

---

## Ship It: `src/rd_tools.py` (10 min)

Here is the finished module. Its design rule is the one from class: an RD number without the
bandwidth that produced it is not a result, so `rd_estimate` returns the bandwidth and the
number of rows on each side with every estimate, `bandwidth_curve` gives the sensitivity table to
report beside it, and `design_checks` puts Diagnosis 2's two tests in one table.

This part is reading only: run the two cells and compare the output with the expected output
below. The first saves the module as a file, the second imports it and runs it on the grant data.

**Reading the cells.** `%%writefile src/rd_tools.py` saves the cell as that file instead of
running it. The notes after a colon or `->` (`outcome: str`, `-> pd.Series`) only say what type
each argument and result is, and `float | None` means a number, or `None`, which lets
`rdrobust` choose the bandwidth. `.N_h` holds the number of rows inside the bandwidth on each
side. `[rd_estimate(...) for h in grid]` builds a list with one result per bandwidth, and
`.astype(...)` turns the counts back into whole numbers. In the second cell, `.round(4)` rounds
every number to four decimals, and `.to_string(index=False)` prints a table without row
numbers.

In [ ]:
%%writefile src/rd_tools.py
"""Regression discontinuity helpers that never report an estimate alone (ECON 5200 Lab 13).

The design rule: every estimate comes back with the bandwidth that produced it, because an RD
number without its bandwidth is one point on a curve the reader cannot see.
"""
import pandas as pd
from rddensity import rddensity
from rdrobust import rdrobust


def rd_estimate(df: pd.DataFrame, outcome: str, running: str, cutoff: float,
                bandwidth: float | None = None) -> pd.Series:
    """Local linear RD, triangular kernel: the estimate, its robust test, h, and n on each side."""
    fit = rdrobust(y=df[outcome], x=df[running], c=cutoff, h=bandwidth)
    return pd.Series({"estimate": fit.coef.loc["Conventional", "Coeff"],
                      "robust_ci_low": fit.ci.loc["Robust", "CI Lower"],
                      "robust_ci_high": fit.ci.loc["Robust", "CI Upper"],
                      "robust_p": fit.pv.loc["Robust", "P>|z|"],
                      "bandwidth": fit.bws.loc["h", "left"],
                      "n_left": fit.N_h[0],
                      "n_right": fit.N_h[1]})


def bandwidth_curve(df: pd.DataFrame, outcome: str, running: str, cutoff: float,
                    grid) -> pd.DataFrame:
    """The estimate at each bandwidth in grid, and the rows it used: the sensitivity table."""
    table = pd.DataFrame([rd_estimate(df, outcome, running, cutoff, h) for h in grid])
    return table[["bandwidth", "estimate", "n_left", "n_right"]].astype({"n_left": int, "n_right": int})


def design_checks(df: pd.DataFrame, running: str, cutoff: float,
                  covariates: list[str]) -> pd.DataFrame:
    """The density test, then an RD with each pre-treatment covariate as the outcome."""
    density = rddensity(X=df[running].to_numpy(dtype=float), c=cutoff)
    rows = [{"test": f"density of {running}", "estimate": float("nan"),     # nan: no estimate here
             "p_value": density.test["p_jk"]}]
    for covariate in covariates:
        result = rd_estimate(df, covariate, running, cutoff)
        rows.append({"test": f"jump in {covariate}", "estimate": result["estimate"],
                     "p_value": result["robust_p"]})
    return pd.DataFrame(rows)

In [ ]:
# GIVEN — run as-is
# Import the module and run it on the grant data: the estimate should match Diagnosis 1
import importlib
import sys

sys.path.insert(0, "src")          # let Python find modules in the src folder
import rd_tools
importlib.reload(rd_tools)         # reload, so a re-run of the %%writefile cell takes effect

main = rd_tools.rd_estimate(rd, "y", "x", 50)
print(f"Estimate {main['estimate']:+.4f}, robust 95% CI [{main['robust_ci_low']:+.4f}, "
      f"{main['robust_ci_high']:+.4f}], bandwidth {main['bandwidth']:.1f}, "
      f"n = {main['n_left']:.0f} left and {main['n_right']:.0f} right of the cutoff")

h_main = main["bandwidth"]
print()
print(rd_tools.bandwidth_curve(rd, "y", "x", 50, [h_main / 2, h_main, 2 * h_main])
      .round(4).to_string(index=False))

# Diagnosis 2's covariates, if you added them: a list of the names that are columns of rd
covariates = [name for name in ["prior_score", "prior_score_jump"] if name in rd]
print()
print(rd_tools.design_checks(rd, "x", 50, covariates).round(4).to_string(index=False))

**Expected output:** an estimate of +0.0789 at a bandwidth of 16.1, as in Diagnosis 1, with a
robust 95% interval of [+0.0466, +0.1139], which is centred on the bias-corrected estimate
rather than on 0.0789, and 639 and 637 students inside the bandwidth on the two sides. The sensitivity table gives 0.0881 at half that bandwidth, 0.0789 at it and 0.0729 at
double. The design checks repeat Diagnosis 2's density test (p-value 0.9645), with one more row
for each of your Diagnosis 2 covariates if you added them.

---

## Extension (Optional — 30 min): Fuzzy RD

When the threshold changes the *probability* of treatment rather than deciding it, RD becomes IV
at the cutoff, which connects this lab directly to the last one.

**The task.** In the cell below, and in new cells under it:
1. Simulate 20,000 units with a running variable spread evenly from 0 to 100 and a cutoff at 50.
   Crossing the cutoff raises the probability of treatment from 0.2 to 0.7, and treatment raises
   the outcome by an amount you choose.
2. Estimate the jump in the outcome at the cutoff (what a sharp RD would report), with
   `rdrobust` as in Diagnosis 1, and the fuzzy RD, with `rdrobust(y=..., x=..., c=50, fuzzy=d)`,
   where `d` is the treatment actually received.
3. Show that the jump in the outcome is about the true effect times the jump in the probability
   of treatment, and answer the questions below.

In [ ]:
# YOUR TASK — optional: fuzzy RD as IV

### Interpretation Questions

1. State the estimand of a fuzzy RD in one sentence.
2. Name the assumption it inherits from Lab 12 that a sharp RD does not need.

*Your answers here:*

1.

2.

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Simulated a grant with a true effect of 8 percentage points at a
  cutoff: a global quadratic estimated [YOUR VALUE], and a local linear
  fit at a data-driven bandwidth of [YOUR VALUE] estimated [YOUR VALUE]
* Repeated the simulation 300 times: the root mean squared error was
  smallest at a bandwidth of [YOUR VALUE]
* Ran a density test for manipulation of the running variable
  (p = [YOUR VALUE] on data where units moved across the cutoff) and a
  covariate continuity test
* Estimated how much more liberally a US House district is represented
  when a Democrat barely wins: [YOUR VALUE] ADA points (Lee, Moretti and
  Butler 2004)
* Wrote my own local linear RD estimator with a triangular kernel, and
  checked it against the rdrobust package

**Please write a README.md entry including:**
1. Project Title: Regression Discontinuity: Bandwidths, Manipulation and Close Elections
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust',
'comprehensive' or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab13-regression-discontinuity`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab13-regression-discontinuity`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab13-regression-discontinuity`, branch `main`,
   commit message `Lab 13 submission`. This pushes the notebook, and the
   notebook's rendered output is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `src/rd_tools.py` module this lab saves is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.